#### Preparation

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import densenet121, DenseNet121_Weights
from torch.utils.data import DataLoader
from PIL import Image
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm

In [2]:
DATA_DIR = r"C:\Users\yugil\Downloads\Cassava_Leaf_Datasets\Cassava_Leaf_Datasets\Classification"

WEIGHTS_DIR = "../weights"

BATCH_SIZE = 32
EPOCHS_FREEZE = 8
EPOCHS_FINETUNE = 20

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [3]:
weights = DenseNet121_Weights.DEFAULT

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])


In [4]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "val"),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)


Classes: ['Healthy', 'Spider Mites', 'leaf blight', 'leaf spot']


In [5]:
print("Train class_to_idx:", train_dataset.class_to_idx)
print("Val class_to_idx:", val_dataset.class_to_idx)

Train class_to_idx: {'Healthy': 0, 'Spider Mites': 1, 'leaf blight': 2, 'leaf spot': 3}
Val class_to_idx: {'Healthy': 0, 'Spider Mites': 1, 'leaf blight': 2, 'leaf spot': 3}


In [6]:
model = densenet121(weights=weights)

in_features = model.classifier.in_features

model.classifier = nn.Linear(in_features, num_classes)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to C:\Users\yugil/.cache\torch\hub\checkpoints\densenet121-a639ec97.pth
100%|██████████| 30.8M/30.8M [00:00<00:00, 40.8MB/s]


DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu

In [7]:
criterion = nn.CrossEntropyLoss()

In [8]:
def train_one_epoch(loader, epoch, epochs, optimizer):
    model.train()
    
    train_loss = 0
    train_preds = []
    train_labels = []

    for images, labels in tqdm(loader, desc=f"Training Epoch {epoch+1}/{epochs}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = outputs.argmax(dim=1)

        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

        train_acc = accuracy_score(train_labels, train_preds)


    return train_loss, train_acc


In [9]:
def validate(loader):
    model.eval()
    
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating "):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = outputs.argmax(dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
        
        val_acc = accuracy_score(val_labels, val_preds)

        precision, recall, f1, _ = precision_recall_fscore_support(
        val_labels, val_preds, average='weighted'
        )   

    return precision, recall, f1, val_loss, val_acc

#### Training 1

In [10]:
for param in model.features.parameters():
    param.requires_grad = False


In [11]:
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-4)


In [12]:

print("\n-----------Starting Phase 1 Training-----------\n")

for epoch in range(EPOCHS_FREEZE):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, EPOCHS_FREEZE, optimizer)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{EPOCHS_FREEZE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n\n")




-----------Starting Phase 1 Training-----------



Validating : 100%|██████████| 8/8 [00:30<00:00,  3.86s/it]



Epoch [1/8]
Train Loss: 40.9555 | Train Acc: 0.3469
Val Loss: 10.6220 | Val Acc: 0.3792
Precision: 0.3762 | Recall: 0.3792 | F1: 0.3526




Validating : 100%|██████████| 8/8 [00:30<00:00,  3.77s/it]



Epoch [2/8]
Train Loss: 37.2696 | Train Acc: 0.4948
Val Loss: 9.8709 | Val Acc: 0.5000
Precision: 0.5307 | Recall: 0.5000 | F1: 0.4876




...


Epoch [6/8]
Train Loss: 26.4622 | Train Acc: 0.8135
Val Loss: 7.5216 | Val Acc: 0.7375
Precision: 0.7639 | Recall: 0.7375 | F1: 0.7405




Validating : 100%|██████████| 8/8 [00:29<00:00,  3.74s/it]



Epoch [7/8]
Train Loss: 24.8510 | Train Acc: 0.8281
Val Loss: 7.0448 | Val Acc: 0.7583
Precision: 0.7703 | Recall: 0.7583 | F1: 0.7592




Validating : 100%|██████████| 8/8 [00:29<00:00,  3.75s/it]


Epoch [8/8]
Train Loss: 23.3610 | Train Acc: 0.8458
Val Loss: 6.7356 | Val Acc: 0.7708
Precision: 0.7815 | Recall: 0.7708 | F1: 0.7728




#### Training 2

In [13]:
for name, param in model.features.named_parameters():
    if "denseblock4" in name:
        param.requires_grad = True


In [14]:
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)


In [15]:

print("\n-----------Starting Phase 2 Training-----------\n")

for epoch in range(EPOCHS_FREEZE):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, EPOCHS_FREEZE, optimizer)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{EPOCHS_FREEZE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n\n")




-----------Starting Phase 2 Training-----------



Validating : 100%|██████████| 8/8 [00:29<00:00,  3.72s/it]



Epoch [1/8]
Train Loss: 14.3881 | Train Acc: 0.8802
Val Loss: 3.0712 | Val Acc: 0.8667
Precision: 0.8662 | Recall: 0.8667 | F1: 0.8663




Validating : 100%|██████████| 8/8 [00:30<00:00,  3.84s/it]



Epoch [2/8]
Train Loss: 7.7317 | Train Acc: 0.9302
Val Loss: 2.4211 | Val Acc: 0.8917
Precision: 0.8943 | Recall: 0.8917 | F1: 0.8915




...


Epoch [6/8]
Train Loss: 2.7933 | Train Acc: 0.9750
Val Loss: 1.6614 | Val Acc: 0.9292
Precision: 0.9323 | Recall: 0.9292 | F1: 0.9293




Validating : 100%|██████████| 8/8 [00:30<00:00,  3.80s/it]



Epoch [7/8]
Train Loss: 2.6608 | Train Acc: 0.9760
Val Loss: 1.6883 | Val Acc: 0.9375
Precision: 0.9458 | Recall: 0.9375 | F1: 0.9381




Validating : 100%|██████████| 8/8 [00:30<00:00,  3.81s/it]


Epoch [8/8]
Train Loss: 2.2686 | Train Acc: 0.9760
Val Loss: 1.5286 | Val Acc: 0.9417
Precision: 0.9445 | Recall: 0.9417 | F1: 0.9419




#### Training 3

In [16]:
for name, param in model.features.named_parameters():
    if "denseblock3" in name:
        param.requires_grad = True


In [17]:
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-5)


In [18]:

print("\n-----------Starting Phase 3 Training-----------\n")

for epoch in range(EPOCHS_FREEZE):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, EPOCHS_FREEZE, optimizer)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{EPOCHS_FREEZE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n\n")




-----------Starting Phase 3 Training-----------



Validating : 100%|██████████| 8/8 [00:30<00:00,  3.87s/it]



Epoch [1/8]
Train Loss: 1.6284 | Train Acc: 0.9854
Val Loss: 1.4621 | Val Acc: 0.9500
Precision: 0.9529 | Recall: 0.9500 | F1: 0.9504




Validating : 100%|██████████| 8/8 [00:31<00:00,  3.90s/it]



Epoch [2/8]
Train Loss: 1.0735 | Train Acc: 0.9948
Val Loss: 1.4234 | Val Acc: 0.9583
Precision: 0.9613 | Recall: 0.9583 | F1: 0.9587




...


Epoch [6/8]
Train Loss: 0.4743 | Train Acc: 1.0000
Val Loss: 1.3002 | Val Acc: 0.9667
Precision: 0.9696 | Recall: 0.9667 | F1: 0.9670




Validating : 100%|██████████| 8/8 [00:30<00:00,  3.83s/it]



Epoch [7/8]
Train Loss: 0.4125 | Train Acc: 0.9990
Val Loss: 1.3312 | Val Acc: 0.9625
Precision: 0.9650 | Recall: 0.9625 | F1: 0.9629




Validating : 100%|██████████| 8/8 [00:30<00:00,  3.81s/it]


Epoch [8/8]
Train Loss: 0.6254 | Train Acc: 0.9969
Val Loss: 1.4533 | Val Acc: 0.9583
Precision: 0.9602 | Recall: 0.9583 | F1: 0.9587




#### Training 4 (Fine-Tuning)

In [19]:
for param in model.parameters():
    param.requires_grad = True


In [20]:
optimizer = optim.Adam(model.parameters(), lr=1e-5)


In [21]:
best_val_acc = 0.0

print("\n-----------Starting Fine-tuning Stage-----------\n")

for epoch in range(EPOCHS_FINETUNE):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, EPOCHS_FINETUNE, optimizer)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{EPOCHS_FINETUNE}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n\n")

        

torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names
}, os.path.join(WEIGHTS_DIR, "DenseNet121.pth"))


-----------Starting Fine-tuning Stage-----------



Validating : 100%|██████████| 8/8 [00:31<00:00,  3.94s/it]



Epoch [1/20]
Train Loss: 0.3867 | Train Acc: 0.9990
Val Loss: 1.2598 | Val Acc: 0.9542
Precision: 0.9563 | Recall: 0.9542 | F1: 0.9547




Validating : 100%|██████████| 8/8 [00:31<00:00,  3.95s/it]



Epoch [2/20]
Train Loss: 0.2643 | Train Acc: 1.0000
Val Loss: 1.3566 | Val Acc: 0.9667
Precision: 0.9696 | Recall: 0.9667 | F1: 0.9670




...


Epoch [18/20]
Train Loss: 0.1782 | Train Acc: 1.0000
Val Loss: 1.3091 | Val Acc: 0.9667
Precision: 0.9696 | Recall: 0.9667 | F1: 0.9670




Validating : 100%|██████████| 8/8 [00:31<00:00,  3.98s/it]



Epoch [19/20]
Train Loss: 0.2014 | Train Acc: 0.9990
Val Loss: 1.4490 | Val Acc: 0.9625
Precision: 0.9674 | Recall: 0.9625 | F1: 0.9631




Validating : 100%|██████████| 8/8 [00:31<00:00,  3.95s/it]


Epoch [20/20]
Train Loss: 0.1375 | Train Acc: 0.9990
Val Loss: 1.3203 | Val Acc: 0.9625
Precision: 0.9652 | Recall: 0.9625 | F1: 0.9628




#### Prediction Sample

In [22]:
checkpoint = torch.load(r'../weights/DenseNet121.pth')

model.load_state_dict(checkpoint["model_state_dict"])
class_names = checkpoint["class_names"]

model.eval()

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]


# Example usage:
prediction = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Healthy\Healthy_val_13.jpg")
prediction1 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Leaf Blight\leaf blight_val_15.jpg")
prediction2 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Leaf Spot\leaf spot_val_14.jpg")
prediction3 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Spider Mites\Spider Mites_val_24.jpg")
print(f"Healthy Predicted class: {prediction}")
print(f"Leaf Blight Predicted class: {prediction1}")
print(f"Leaf Spot Predicted class: {prediction2}")
print(f"Spider Mites Predicted class: {prediction3}")

C:\Users\yugil\AppData\Local\Temp\ipykernel_3624\2528340374.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(r'../weights/DenseNet121.pth')


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\johnp\\Desktop\\Dataset\\Cassava_Leaf_Datasets\\Classification\\val\\Healthy\\Healthy_val_13.jpg'